In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

#### ※ `WHERE句`のサブクエリでよく使用される演算子

サブクエリの結果数は`行数`を基準にする。  
列数は問わず、複数の列があっても、1行を1件の結果として扱う。

| 演算子 | サブクエリの結果 | 使用目的 |
|---|---|---|
| `=` | 1件 | 等しい値を比較する |
| `>`、`<`、`>=`、`<=` | 1件 | 大小を比較する |
| `IN` | 複数件 | 一覧に含まれるか確認する |
| `NOT IN` | 複数件 | 一覧から除外する |
| `EXISTS` | 件数は問わない | データが存在するか確認する |
| `NOT EXISTS` | 件数は問わない | データが存在しないか確認する |

## 1. `=`（等しい）

サブクエリが1つの値だけを返す場合に使用する。

#### 例：平均レンタル料金と同じ映画を取得する

In [2]:
%%sql

SELECT title, rental_rate
FROM film
WHERE rental_rate =
(
    SELECT AVG(rental_rate)
    FROM film
);

title,rental_rate


## 2. `>`、`<`、`>=`、`<=`

サブクエリが1つの値を返す場合に使用する。

#### 例：平均レンタル料金より高い映画を取得する

In [4]:
%%sql

SELECT
    title,
    rental_rate
FROM film
WHERE rental_rate > (
    SELECT AVG(rental_rate)
    FROM film
)

LIMIT 5;

title,rental_rate
ACE GOLDFINGER,4.99
ADAPTATION HOLES,2.99
AFFAIR PREJUDICE,2.99
AFRICAN EGG,2.99
AGENT TRUMAN,2.99


## 3. IN

サブクエリが複数の値を返す場合に使用する。

#### 例：`Action`または`Comedy`カテゴリの映画を取得する

In [5]:
%%sql

SELECT
    film_id,
    category_id
FROM film_category
WHERE category_id IN (
    -- サブクエリの結果：2行以上
    SELECT category_id
    FROM category
    WHERE name IN ('Action', 'Comedy')
);

film_id,category_id
19,1
21,1
29,1
38,1
56,1
67,1
97,1
105,1
111,1
115,1


## 4. NOT IN

サブクエリの結果に含まれないデータを取得する。

#### 例：`Action`と`Comedy`を除くカテゴリの映画を取得する

In [6]:
%%sql

SELECT
    film_id,
    category_id
FROM film_category
WHERE category_id NOT IN (
    -- サブクエリの結果：2行以上
    SELECT category_id
    FROM category
    WHERE name IN ('Action', 'Comedy')
);

film_id,category_id
18,2
23,2
36,2
70,2
78,2
89,2
118,2
121,2
134,2
154,2


## 5. EXISTS

`EXISTS`は、メインクエリの各行に対応するデータが、サブクエリに存在するかどうかを確認する `(INNER JOIN と似ている)`

サブクエリの結果が1行でも存在すれば`TRUE`となり、そのメインクエリの行を取得する。

#### 例：映画が1本以上登録されているカテゴリのみを取得する

In [9]:
%%sql

SELECT *
FROM category c
WHERE EXISTS
(
    SELECT *
    FROM film_category fc
    WHERE fc.category_id = c.category_id
);

category_id,name,last_update
1,Action,2006-02-15 04:46:27
2,Animation,2006-02-15 04:46:27
3,Children,2006-02-15 04:46:27
4,Classics,2006-02-15 04:46:27
5,Comedy,2006-02-15 04:46:27
6,Documentary,2006-02-15 04:46:27
7,Drama,2006-02-15 04:46:27
8,Family,2006-02-15 04:46:27
9,Foreign,2006-02-15 04:46:27
10,Games,2006-02-15 04:46:27


#### # 以下のJOINと結果は同じである。

In [7]:
%%sql

SELECT c.*
FROM category c
INNER JOIN film_category fc
    ON fc.category_id = c.category_id;

category_id,name,last_update
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27
1,Action,2006-02-15 04:46:27


#### ※ `EXISTS`と`INNER JOIN`は、関連データが存在するカテゴリを取得するという目的は似ている。